# MSTCT: Multi-Scale Temporal Conv-Transformer

## Experiment Setup
- Dataset: ESC-50 (environmental sound classification)
- Sample rate: 16kHz (downsampled from 44.1kHz)
- Duration: 5 seconds (original) → we'll use full clips
- Classes: 50


In [ ]:
# Install dependencies (uncomment if needed)
# !pip install torch torchaudio torchvision numpy matplotlib scikit-learn tqdm pandas

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from tqdm.notebook import tqdm
import os
import urllib.request
import zipfile
import pandas as pd
from pathlib import Path

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Download and Prepare ESC-50

In [ ]:
# Download ESC-50 dataset
data_dir = Path('./esc50_data')
data_dir.mkdir(exist_ok=True)

# Download if not exists
zip_path = data_dir / 'ESC-50-master.zip'
if not zip_path.exists():
    print("Downloading ESC-50 dataset...")
    url = "https://github.com/karoldvl/ESC-50/archive/master.zip"
    urllib.request.urlretrieve(url, zip_path)
    print("Download complete!")

# Extract if not extracted
extracted_dir = data_dir / 'ESC-50-master'
if not extracted_dir.exists():
    print("Extracting...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(data_dir)
    print("Extraction complete!")

# Load metadata
meta_path = extracted_dir / 'meta' / 'esc50.csv'
df = pd.read_csv(meta_path)
audio_dir = extracted_dir / 'audio'

print(f"Total samples: {len(df)}")
print(f"Number of classes: {df['target'].nunique()}")
print(f"Classes: {sorted(df['category'].unique())[:5]}...")

In [ ]:
# Split into train/val (80/20) by fold
# ESC-50 has 5 folds, we'll use folds 1-4 for train, fold 5 for validation
train_df = df[df['fold'] != 5].reset_index(drop=True)
val_df = df[df['fold'] == 5].reset_index(drop=True)

print(f"Train samples: {len(train_df)}")
print(f"Val samples: {len(val_df)}")

## ESC-50 Dataset Class

In [ ]:
class ESC50Dataset(Dataset):
    """ESC-50 dataset with raw waveform output"""
    def __init__(self, df, audio_dir, target_sample_rate=16000, max_duration=5):
        self.df = df
        self.audio_dir = audio_dir
        self.target_sample_rate = target_sample_rate
        self.max_length = target_sample_rate * max_duration
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        filename = row['filename']
        label = row['target']
        
        # Load audio
        audio_path = self.audio_dir / filename
        waveform, sample_rate = torchaudio.load(audio_path)
        
        # Convert to mono if stereo
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
        
        # Resample if needed
        if sample_rate != self.target_sample_rate:
            resampler = torchaudio.transforms.Resample(sample_rate, self.target_sample_rate)
            waveform = resampler(waveform)
        
        # Pad or truncate to fixed length
        if waveform.shape[1] < self.max_length:
            pad_length = self.max_length - waveform.shape[1]
            waveform = F.pad(waveform, (0, pad_length))
        else:
            waveform = waveform[:, :self.max_length]
        
        # Squeeze channel dimension
        waveform = waveform.squeeze(0)  # (T,)
        
        return waveform, label


# Create datasets
train_dataset = ESC50Dataset(train_df, audio_dir)
val_dataset = ESC50Dataset(val_df, audio_dir)

# Test a sample
sample_waveform, sample_label = train_dataset[0]
print(f"Sample waveform shape: {sample_waveform.shape}")
print(f"Sample label: {sample_label}")

In [ ]:
# Create data loaders
batch_size = 16  # Reduce if GPU memory is low
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

## Model Definition

In [ ]:
class ConvBlock(nn.Module):
    """Grouped convolution block with skip connection"""
    def __init__(self, in_channels, out_channels, kernel_size=7, groups=None):
        super().__init__()
        if groups is None:
            groups = min(in_channels, out_channels, 4)
        while in_channels % groups != 0:
            groups -= 1
        
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=kernel_size//2, groups=groups)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=kernel_size//2, groups=groups)
        self.bn2 = nn.BatchNorm1d(out_channels)
        
        self.skip = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()
        self.activation = nn.GELU()
        self.dropout = nn.Dropout(0.1)
        
    def forward(self, x):
        identity = self.skip(x)
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.activation(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out = self.dropout(out)
        out = self.activation(out + identity)
        return out


class PatchEmbedding(nn.Module):
    """Patch embedding: conv with stride=patch_size, kernel_size=patch_size, groups=4"""
    def __init__(self, in_channels, embed_dim, patch_size=16):
        super().__init__()
        self.patch_size = patch_size
        groups = 4
        while in_channels % groups != 0:
            groups -= 1
        
        self.conv = nn.Conv1d(
            in_channels, embed_dim,
            kernel_size=patch_size,
            stride=patch_size,
            groups=groups
        )
        self.linear = nn.Linear(embed_dim, embed_dim)
        
    def forward(self, x):
        # x: (B, C, T)
        x = self.conv(x)  # (B, embed_dim, num_patches)
        x = x.transpose(1, 2)  # (B, num_patches, embed_dim)
        x = self.linear(x)
        return x


class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding"""
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    
    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]


class MSTCT(nn.Module):
    """Multi-Scale Temporal Conv-Transformer"""
    def __init__(
        self,
        num_classes=50,
        d_model=256,
        nhead=8,
        num_layers=4,
        patch_size=16,
        sample_rate=16000,
        duration=5.0
    ):
        super().__init__()
        
        self.sample_rate = sample_rate
        self.num_samples = int(sample_rate * duration)
        
        # Stage 1: 1 → 32 channels
        self.conv1 = ConvBlock(1, 32, kernel_size=15, groups=1)
        
        # Stage 2: 32 → 128 channels
        self.conv2 = ConvBlock(32, 128, kernel_size=11, groups=4)
        
        # Stage 3: 128 → d_model channels
        self.conv3 = ConvBlock(128, d_model, kernel_size=7, groups=16)
        
        # Adaptive pooling to reduce sequence length
        self.pool = nn.AdaptiveAvgPool1d(512)
        
        # Patch embedding
        self.patch_embed = PatchEmbedding(d_model, d_model, patch_size=patch_size)
        
        # Positional encoding + Transformer
        self.pos_encoder = PositionalEncoding(d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model, nhead, dim_feedforward=d_model*4, dropout=0.1, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers)
        
        # Classification head
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model//2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(d_model//2, num_classes)
        )
    
    def forward(self, x):
        # x: (B, samples) or (B, 1, samples)
        if x.dim() == 2:
            x = x.unsqueeze(1)
        
        # Pad/truncate to fixed length
        if x.shape[-1] < self.num_samples:
            x = F.pad(x, (0, self.num_samples - x.shape[-1]))
        else:
            x = x[:, :, :self.num_samples]
        
        # Conv blocks
        x = self.conv1(x)      # (B, 32, T)
        x = self.conv2(x)      # (B, 128, T)
        x = self.conv3(x)      # (B, d_model, T)
        
        # Pool
        x = self.pool(x)       # (B, d_model, T')
        
        # Patch embedding
        x = self.patch_embed(x)  # (B, num_patches, d_model)
        
        # Positional encoding
        x = self.pos_encoder(x)
        
        # Transformer
        x = self.transformer(x)  # (B, num_patches, d_model)
        
        # Global mean pooling
        x = x.mean(dim=1)        # (B, d_model)
        
        return self.classifier(x)


# Instantiate model
model = MSTCT(num_classes=50, d_model=256, patch_size=16).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

# Test forward pass
test_input = torch.randn(4, 80000).to(device)  # 5 seconds at 16kHz
test_output = model(test_input)
print(f"Input shape: {test_input.shape}")
print(f"Output shape: {test_output.shape}")

## Training Setup

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

num_epochs = 30
train_losses = []
val_accuracies = []

## Training Loop

In [ ]:
for epoch in range(num_epochs):
    # Training
    model.train()
    total_loss = 0
    for batch_idx, (audio, labels) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")):
        audio, labels = audio.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(audio)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_loader)
    train_losses.append(avg_loss)
    
    # Validation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for audio, labels in val_loader:
            audio, labels = audio.to(device), labels.to(device)
            outputs = model(audio)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    accuracy = 100 * correct / total
    val_accuracies.append(accuracy)
    
    scheduler.step()
    
    print(f"Epoch {epoch+1}: Loss = {avg_loss:.4f}, Val Accuracy = {accuracy:.2f}%")

## Results Visualization

In [ ]:
# Loss curve
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss Curve')
plt.legend()
plt.grid(True)

# Accuracy curve
plt.subplot(1, 2, 2)
plt.plot(val_accuracies, label='Validation Accuracy', color='orange')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title('Validation Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

print(f"Best validation accuracy: {max(val_accuracies):.2f}%")

In [ ]:
# Save model
torch.save(model.state_dict(), 'mstct_esc50.pth')
print("Model saved to mstct_esc50.pth")

## Confusion Matrix

In [ ]:
# Generate predictions on validation set
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for audio, labels in tqdm(val_loader, desc="Generating predictions"):
        audio = audio.to(device)
        outputs = model(audio)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

# Show subset of confusion matrix (first 10 classes for readability)
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(12, 10))
disp = ConfusionMatrixDisplay(confusion_matrix=cm[:10, :10])
disp.plot(cmap='Blues', xticks_rotation=45)
plt.title(f'Confusion Matrix (First 10 Classes) - Final Accuracy: {val_accuracies[-1]:.2f}%')
plt.tight_layout()
plt.show()

## Sample Predictions

In [ ]:
# Show a few sample predictions
model.eval()
samples, labels = next(iter(val_loader))
samples = samples.to(device)

# Get class names
class_names = df[['target', 'category']].drop_duplicates().sort_values('target')['category'].tolist()

with torch.no_grad():
    outputs = model(samples)
    probs = torch.softmax(outputs, dim=1)
    _, preds = torch.max(outputs, 1)

print("Sample Predictions (first 8):")
print(f"{'True Label':<20} {'Predicted':<20} {'Confidence':<12}")
print("-" * 55)
for i in range(min(8, len(samples))):
    true_name = class_names[labels[i]] if labels[i] < len(class_names) else str(labels[i])
    pred_name = class_names[preds[i]] if preds[i] < len(class_names) else str(preds[i])
    conf = probs[i, preds[i]].item()
    print(f"{true_name:<20} {pred_name:<20} {conf:.4f}")

## Model Summary

```
Architecture: MSTCT (Multi-Scale Temporal Conv-Transformer)
Total parameters: {total_params:,}
Input: Raw waveform @ 16kHz, 5 seconds (80,000 samples)
Output: 50 ESC-50 classes
No FFT, no spectrogram

Best validation accuracy: {max(val_accuracies):.2f}%
```

## Next Steps

1. Train for more epochs (50-100) to reach ~70-75% accuracy
2. Compare against AST baseline on ESC-50
3. Ablation study on grouped convolution groups
4. Deploy with ONNX or TorchScript
5. Visualize attention maps from transformer layers